## ZeroShot evaluation on BanglaCalamityMMD dataset

### Load dataset from drive

In [ ]:
#Define paths
zip_path = '/content/drive/MyDrive/BanglaCalamityMMD.zip'   # ← Update this path
extract_path = '/content/BanglaCalamityMMD'

#Extract using 7z
!apt-get install -y p7zip-full > /dev/null 2>&1

import os
if not os.path.exists(extract_path):
    print("Extracting the dataset... (this may take 5-15 minutes the first time)")
    !7z x "{zip_path}" -o"{extract_path}" -y
else:
    print("Dataset already extracted. Skipping extraction.")

In [ ]:
# Check if loaded correctly
import os
print("Files/folders in dataset root:", os.listdir("/content/BanglaCalamityMMD"))

### Format Dataset: Text and Image paths

In [ ]:
import pandas as pd
import os
from PIL import Image

def load_test_data(dataset_root="/content/BanglaCalamityMMD"):
    # Add paths of testing files for text and images
    csv_path = os.path.join(dataset_root, "BanglaCalamityMMD/Disaster_test.csv")
    images_dir = os.path.join(dataset_root, "BanglaCalamityMMD/Test")

    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV not found at: {csv_path}")
    if not os.path.exists(images_dir):
        raise FileNotFoundError(f"Test images folder not found at: {images_dir}")

    df = pd.read_csv(csv_path)
    print("Original CSV columns:", df.columns.tolist())

    # Standardize column names
    df = df.rename(columns={
        "image_id": "image_filename",
        "context": "caption",
        "category": "true_label"
    })

    def get_image_path(row):
        filename = str(row["image_filename"]).strip()
        base_path = os.path.join(images_dir, filename)

        # Image path neyar shomoy extension ta add kora chhilo na oita kortesi ekhane
        # Common extensions for images (this is to avoid any errors of FileType)
        for ext in [".jpg", ".jpeg", ".png", ""]:
            full_path = base_path + ext
            if os.path.exists(full_path):
                return full_path
        # If nothing works, return original (for debugging)
        return base_path

    df["image_path"] = df.apply(get_image_path, axis=1)

    # Verify a few paths
    print("\nExample corrected image path:", df.iloc[0]["image_path"])
    print("Does first image exist?", os.path.exists(df.iloc[0]["image_path"]))

    # Count missing files
    missing = df[~df["image_path"].apply(os.path.exists)]
    if len(missing) > 0:
        print(f"⚠️  Warning: {len(missing)} images not found!")
    else:
        print("✅ All image paths are valid!")

    return df

# Run the loader
test_df = load_test_data()

print("\nTest samples loaded:", len(test_df))
print(test_df[["true_label"]].value_counts())
print("\nExample row:\n", test_df.iloc[0])

### Parse Prediction
Predictions might be in Bangla but the labels are in english, so we map the predictions to the right label for maximum accuracy

In [ ]:
import re

CLASSES = ["Earthquake", "Flood", "Landslides", "Wildfire", "Tropical Storm", "Drought", "Human Damage", "Non Disaster"]

def parse_prediction(response: str):
    if not isinstance(response, str):
        return "Non Disaster"

    response_lower = response.lower().strip()
    response_orig = response.strip()

    # 1. English class names (in case the predictions are in english)
    class_map = {
        "earthquake": "Earthquake",
        "flood": "Flood",
        "landslide": "Landslides",
        "landslides": "Landslides",
        "hillslide": "Landslides",
        "mudslide": "Landslides",
        "wildfire": "Wildfire",
        "fire": "Wildfire",
        "tropical storm": "Tropical Storm",
        "tornado": "Tropical Storm",
        "storm": "Tropical Storm",
        "cyclone": "Tropical Storm",
        "hurricane": "Tropical Storm",
        "typhoon": "Tropical Storm",
        "drought": "Drought",
        "water": "Drought",
        "human damage": "Human Damage",
        "war": "Human Damage",
        "bombing": "Human Damage",
        "explosion": "Human Damage",
        "bomb": "Human Damage",
        "attack": "Human Damage",
        "destruction": "Human Damage",
        "destroy": "Human Damage",
        "collapse": "Human Damage",
        "damage": "Human Damage",
        "non disaster": "Non Disaster",
        "normal": "Non Disaster",
        "no disaster": "Non Disaster"
    }

    for eng, label in class_map.items():
        if eng in response_lower:
            return label

    # 2. Bangla terms (usually the predictions will be in Bangla)
    bangla_map = {
        "ভূমিকম্প": "Earthquake",
        "ভূমিকম্পের": "Earthquake",
        "বন্যা": "Flood",
        "যমুনা": "Flood",
        "বৃষ্টি": "Flood",
        "বন্যার": "Flood",
        "ভূমিধস": "Landslides",
        "ভূমিধস": "Landslides",
        "ভূমিসৃতি": "Landslides",
        "ভূমিধ্বস": "Landslides",
        "ভূমি ক্ষয়": "Landslides",
        "মাডস্লাইড": "Landslides",
        "হিলস্লাইড": "Landslides",
        "দাবানল": "Wildfire",
        "অগ্নিকাণ্ড": "Wildfire",
        "বনের আগুন": "Wildfire",
        "ঘূর্ণিঝড়": "Tropical Storm",
        "টর্নেডো": "Tropical Storm",
        "হারিকেন": "Tropical Storm",
        "ঝড়": "Tropical Storm",
        "ঘূর্ণাবর্ত": "Tropical Storm",
        "ট্রপিক্যাল": "Tropical Storm",
        "খরা": "Drought",
        "জলের অভাবে": "Drought",
        "জল নাই": "Drought",
        "শুষ্কজল": "Drought",
        "বৃষ্টি নেই": "Drought",
        "ক্ষুধা": "Drought",
        "পানি কম": "Drought",
        "জল সংকট": "Drought",
        "পানির ঘাটতি": "Drought",
        "অভাবে": "Drought",
        "শুষ্ক": "Drought",
        "মানুষের ক্ষতি": "Human Damage",
        "ধ্বংস": "Human Damage",
        "মানবসৃষ্ট": "Human Damage",
        "মানব": "Human Damage",
        "বিস্ফোরণ": "Human Damage",
        "বিষক্রিয়া": "Human Damage",
        "দুর্ঘটনা": "Human Damage",
        "যুদ্ধ": "Human Damage",
        "লড়াই": "Human Damage",
        "হত্যা": "Human Damage",
        "ধর্মঘট": "Human Damage",
        "বোমা": "Human Damage",
        "হামলা": "Human Damage",
        "আক্রমণ": "Human Damage",
    }

    for bangla, label in bangla_map.items():
        if bangla in response_orig:   # keep original case for Bangla
            return label

    # 3. Fallback: if the response is very short or contains the class in any form
    for cls in CLASSES:
        if cls.lower() in response_lower:
            return cls

    return "Non Disaster"

# Eta emni banay rakhsilam, use kora hoy nai kothao
def evaluate(predictions, true_labels):
    correct = sum(1 for p, t in zip(predictions, true_labels) if p == t)
    accuracy = correct / len(predictions) * 100
    print(f"Zero-Shot Accuracy: {accuracy:.2f}%")
    return accuracy

### Evaluation Function

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
import os

CLASSES = ["Earthquake", "Flood", "Landslides", "Wildfire", "Tropical Storm", "Drought", "Human Damage", "Non Disaster"]

def evaluate_model(predictions, true_labels, model_name="Model", save_dir="results"):
    os.makedirs(save_dir, exist_ok=True)

    y_true = np.array(true_labels)
    y_pred = np.array(predictions)

    # (kaeno jani class er class r label number mismatch error ashe tai eita kora)
    # === Force everything to be one of the 8 classes ===
    valid_classes = set(CLASSES)
    y_pred = np.array([pred if pred in valid_classes else "Non Disaster" for pred in y_pred])

    print(f"\n=== {model_name} Zero-Shot Evaluation ===")
    print(f"Total samples : {len(y_true)}")

    # Count distribution of predictions
    print("\nPredicted class distribution:")
    print(pd.Series(y_pred).value_counts())

    # Metrics
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

    print(f"\nAccuracy           : {acc*100:.2f}%")
    print(f"Macro Precision    : {prec*100:.2f}%")
    print(f"Macro Recall       : {rec*100:.2f}%")
    print(f"Macro F1-score     : {f1*100:.2f}%")

    # Classification Report
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred,
                                labels=CLASSES,           # Force exactly 8 classes
                                target_names=CLASSES,
                                digits=4,
                                zero_division=0))

    # Confusion Matrix (8x8 clean)
    cm = confusion_matrix(y_true, y_pred, labels=CLASSES)

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES)
    plt.title(f'Confusion Matrix - {model_name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f"{save_dir}/{model_name}_confusion_matrix.png", dpi=300, bbox_inches='tight')
    plt.show()

    # Save results
    results_df = pd.DataFrame({
        "true_label": y_true,
        "predicted": y_pred,
        "raw_response": test_df.get("blip2_raw_response", [""]*len(test_df))
    })
    results_df.to_csv(f"{save_dir}/{model_name}_predictions.csv", index=False)

    print(f"\n✅ Results saved in ./{save_dir}/")
    return acc, prec, rec, f1

### Memory clear

Before loading and running any heavy model, run this to clear out cache from processor RAM. This makes sure noy OutofMemory error occurs.

While running Qwen, I was bombarded with this error. The GPU RAM was constantly running out of memory. The simple clear method in the main function was not working. So I shifted to a lighter version of Qwen (used 2B instead of 7B) and opted for this aggressive memory clear function.

In [ ]:
import torch
import gc

# Very aggressive memory clear
torch.cuda.empty_cache()
gc.collect()
torch.cuda.empty_cache()

# Optional: Reset CUDA cache
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.reset_accumulated_memory_stats()

print("Memory cleared. Current GPU memory:", torch.cuda.memory_allocated()/1024**3, "GB")

### BLIP2

**Most Challenging Part:** Improving accuracy by better parsing.

In [ ]:
# ====================== INSTALL & IMPORT ======================
!pip install -q transformers torch accelerate

import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from tqdm import tqdm
import gc

# Clear memory
torch.cuda.empty_cache()
gc.collect()

# ======================= LOAD BLIP-2 ==========================
processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
model = Blip2ForConditionalGeneration.from_pretrained("Salesforce/blip2-opt-2.7b", torch_dtype=torch.float16, device_map="auto")

# ======================= PROMPT =======================
PROMPT = "এই ছবিটি এবং টেক্সটটি দেখে বলুন এটি কোন ধরণের দুর্যোগ।"

# ====================== ZERO-SHOT FUNCTION ======================
def blip2_zero_shot(df):
    preds = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        image = Image.open(row["image_path"]).convert("RGB")
        text_input = f"{PROMPT}\nছবির টেক্সট: {row['caption']}"

        inputs = processor(images=image, text=text_input, return_tensors="pt").to(model.device)
        generated_ids = model.generate(**inputs, max_new_tokens=50)
        response = processor.decode(generated_ids[0], skip_special_tokens=True)
        preds.append(parse_prediction(response))
        raw_responses.append(response)
    return preds

# ====================== RUN EVALUATION ======================
predictions = blip2_zero_shot(test_df)
blip2_predictions = predictions

# Evaluate
blip2_acc, blip2_prec, blip2_rec, blip2_f1 = evaluate_model(
    predictions=predictions,
    true_labels=test_df["true_label"].tolist(),
    model_name="BLIP-2_ZeroShot"
)

In [ ]:
blip2_acc, blip2_prec, blip2_rec, blip2_f1 = evaluate_model(
    predictions=predictions,
    true_labels=test_df["true_label"].tolist(),
    model_name="BLIP-2_ZeroShot"
)

### Qwen2-VL-2B

**Most Challenging part:** The constant OutOfMemory errors

In [ ]:
!pip install qwen-vl-utils

In [ ]:
# ====================== INSTALL & IMPORT ======================
!pip install -q transformers accelerate bitsandbytes

import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info
from tqdm import tqdm
import gc

# Clear GPU memory
torch.cuda.empty_cache()
gc.collect()

# ====================== LOAD Qwen2-VL-2B ======================
print("Loading Qwen2-VL-2B in 4-bit...")

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",           # ← 2B version
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True
)

processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    trust_remote_code=True
)

print("✅ Qwen2-VL-2B loaded successfully!")
print(f"GPU Memory Used: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

# ====================== PROMPT ======================
PROMPT = "এই ছবিটি এবং টেক্সটটি দেখে বলুন এটি কোন ধরণের দুর্যোগ।"

# ====================== ZERO-SHOT FUNCTION ======================
def qwen2vl_zero_shot(df):
    preds = []
    print("Running Qwen2-VL-2B Zero-Shot...")

    for _, row in tqdm(df.iterrows(), total=len(df)):
        try:
            messages = [
                {"role": "user", "content": [
                    {"type": "image", "image": row["image_path"]},
                    {"type": "text", "text": f"{PROMPT}\nছবির টেক্সট: {row['caption']}"}
                ]}
            ]

            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            image_inputs, _ = process_vision_info(messages)

            inputs = processor(
                text=[text],
                images=image_inputs,
                padding=True,
                return_tensors="pt"
            ).to(model.device)

            generated_ids = model.generate(
                **inputs,
                max_new_tokens=50,
                do_sample=False,
                temperature=0.0
            )

            response = processor.decode(generated_ids[0], skip_special_tokens=True)
            parsed = parse_prediction(response)
            preds.append(parsed)

        except Exception as e:
            print(f"Error on one image: {e}")
            preds.append("Non-Disaster")

    return preds


# ====================== RUN EVALUATION ======================
qwen2vl_predictions = qwen2vl_zero_shot(test_df)

# Evaluate
qwen2vl_acc, qwen2vl_prec, qwen2vl_rec, qwen2vl_f1 = evaluate_model(
    predictions=qwen2vl_predictions,
    true_labels=test_df["true_label"].tolist(),
    model_name="Qwen2-VL-2B_ZeroShot"
)

### LLaVA-1.5-7B

**Most Challenging part:** The version mismatches internally. So Llava here needs a lower version of transformers, which clashes with the version of AutoProcessor. While fixing both their versions, the pytorch version mismatches with that. After fixing all the 3 versions, that is forcefully re-installing the modules though it was already in colab, it clashes with, you guessed it, Colab itself.

It was a journey of it's own, but somehow I fixed it and here are the code blocks. A little disoriented than the other models.

In [ ]:
!pip install -U "transformers>=4.39.0"
!pip install peft bitsandbytes
!pip install -U "trl>=0.8.3"

In [ ]:
import torch
from transformers import AutoTokenizer, AutoProcessor, TrainingArguments, LlavaForConditionalGeneration, BitsAndBytesConfig
from trl import SFTTrainer
from peft import LoraConfig

In [ ]:
model_id = "llava-hf/llava-1.5-7b-hf"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
)

model = LlavaForConditionalGeneration.from_pretrained(model_id,
                                                      quantization_config=quantization_config,
                                                      torch_dtype=torch.float16)

print("✅ LLaVA-1.5-7B loaded!")

In [ ]:
processor = AutoProcessor.from_pretrained(model_id)

In [ ]:
from tqdm import tqdm
from PIL import Image

# ====================== PROMPT ======================
PROMPT = "এই ছবিটি এবং টেক্সটটি দেখে বলুন এটি কোন ধরণের দুর্যোগ।"

# ====================== ZERO-SHOT FUNCTION ======================
def llava_zero_shot(df):
    preds = []
    print("Running LLaVA-v1.6 Zero-Shot...")

    for _, row in tqdm(df.iterrows(), total=len(df)):
        try:
            # Load image
            image = Image.open(row["image_path"]).convert("RGB")

            # Prepare conversation
            conversation = [
                {"role": "user", "content": [
                    {"type": "text", "text": f"{PROMPT}\nছবির টেক্সট: {row['caption']}"},
                    {"type": "image", "image": image}
                ]}
            ]

            prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)

            # Process inputs
            inputs = processor(
                images=image,
                text=prompt,
                return_tensors="pt"
            ).to(model.device)

            # Generate
            output = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                temperature=0.0
            )

            response = processor.decode(output[0], skip_special_tokens=True)
            parsed = parse_prediction(response)
            preds.append(parsed)

        except Exception as e:
            print(f"Error on one image: {e}")
            preds.append("Non-Disaster")

    return preds


# ====================== RUN ======================
llava_predictions = llava_zero_shot(test_df)

# Evaluate
llava_acc, llava_prec, llava_rec, llava_f1 = evaluate_model(
    predictions=llava_predictions,
    true_labels=test_df["true_label"].tolist(),
    model_name="LLaVA-v1.6_ZeroShot"
)

In [ ]:
# Evaluate
llava_acc, llava_prec, llava_rec, llava_f1 = evaluate_model(
    predictions=llava_predictions,
    true_labels=test_df["true_label"].tolist(),
    model_name="LLaVA-v1.6_ZeroShot"
)

### Gemma-3-4B

Major Challenge: The API token

In [ ]:
from huggingface_hub import login

login("HUGGINGFACE TOKEN")

In [ ]:
# ====================== 1. INSTALL ======================
!pip install -q transformers accelerate bitsandbytes

import torch
from transformers import AutoProcessor, Gemma3ForConditionalGeneration, BitsAndBytesConfig
from tqdm import tqdm
from PIL import Image
import gc

# Clear memory
torch.cuda.empty_cache()
gc.collect()

# ====================== 2. LOAD GEMMA 3 ======================
print("Loading Gemma-3-4B-IT...")

model_id = "google/gemma-3-4b-it"

# 4-bit Config
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

model = Gemma3ForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=quantization_config,
    trust_remote_code=True
)

print("✅ Gemma 3 loaded successfully!")

# ====================== 3. PROMPT ======================
PROMPT = "এই ছবিটি এবং টেক্সটটি দেখে বলুন এটি কোন ধরণের দুর্যোগ।"

# ====================== 4. ZERO-SHOT FUNCTION ======================
def gemma3_zero_shot(df):
    preds = []
    print("Running Gemma-3 Zero-Shot Evaluation...")

    for _, row in tqdm(df.iterrows(), total=len(df)):
        try:
            image = Image.open(row["image_path"]).convert("RGB")

            # Prepare messages
            messages = [{
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": f"{PROMPT}\nছবির টেক্সট: {row['caption']}"}
                ]
            }]

            # Process inputs
            inputs = processor.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=True,
                return_tensors="pt"
            ).to(model.device)

            # Generate
            generated_ids = model.generate(
                inputs,
                max_new_tokens=80,
                do_sample=False,
                temperature=0.0
            )

            response = processor.decode(generated_ids[0], skip_special_tokens=True)
            parsed = parse_prediction(response)
            preds.append(parsed)

        except Exception as e:
            print(f"Error on one image: {e}")
            preds.append("Non-Disaster")

    return preds


# ====================== 5. RUN ======================
gemma_predictions = gemma3_zero_shot(test_df)

# Evaluate
gemma_acc, gemma_prec, gemma_rec, gemma_f1 = evaluate_model(
    predictions=gemma_predictions,
    true_labels=test_df["true_label"].tolist(),
    model_name="Gemma-3-4B_ZeroShot"
)